In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

##Load silver product

In [0]:
df_silver = spark.table(SILVER_PRODUCT)
print(f"Silver rows: {df_silver.count()}")
df_silver.show(3, truncate=40)

##Build dim_product

In [0]:
df_dim_product = (df_silver

    # Discount logic
    .withColumn("is_price_change",
        F.when(
            (F.col("before_price_sar").isNotNull()) &
            (F.col("before_price_sar") > 0) &
            (F.col("before_price_sar") > F.col("current_price_sar")),
            True
        ).otherwise(False)
    )
    .withColumn("price_change_pct",
        F.when(
            F.col("is_price_change") == True,
            F.round(
                (F.col("before_price_sar") - F.col("current_price_sar")) /
                F.col("before_price_sar") * 100, 2
            )
        ).otherwise(F.lit(0.0))
    )

    # Add ingestion metadata
    .withColumn("_gold_ingested_at",
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "asin",
        "product_title",
        "category",
        "current_price_sar",
        "before_price_sar",
        "is_price_change",
        "price_change_pct",
        "rating",
        "total_reviews",
        "is_amazon_choice",
        "is_best_seller",
        "is_prime",
        "is_sponsored",
        "bought_past_month",
        "position",
        "stock_warning",
        "product_photo",
        "product_url",
        "_snapshot_date",
        "_gold_ingested_at"
    )
)

print(f"dim_product rows: {df_dim_product.count()}")
display(df_dim_product)

##Write to Gold

In [0]:
(df_dim_product
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_PRODUCT)
)

print(f"✅ {df_dim_product.count()} rows written to {GOLD_PRODUCT}")

##Validate

In [0]:
df_check = spark.table(GOLD_PRODUCT)

print(f"Total rows       : {df_check.count()}")
print(f"Discounted       : {df_check.filter(F.col('is_price_change') == True).count()}")
print(f"Avg discount pct : {df_check.filter(F.col('is_price_change') == True).agg(F.round(F.avg('price_change_pct'), 2)).collect()[0][0]}%")
print(f"Categories       : {[r[0] for r in df_check.select('category').distinct().collect()]}")

df_check.select(
    "asin", "product_title", "category",
    "current_price_sar", "is_price_change", "price_change_pct","rating"
).show(5, truncate=40)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_product;